# PrivAware v2 — Day 1: Phishing Dataset

**Goal for today:** Load the PhiUSIIL phishing URL dataset, clean it, explore it, and split it into train/test sets ready for model training tomorrow.

**Time:** ~45 minutes to run everything

**Before you start:**
- Make sure you are signed into your Google account
- Go to Runtime → Change runtime type → Select **T4 GPU** → Save
- Then run each cell one by one (Shift+Enter)


## Step 1 — Install required libraries
Run this first. It will take about 2 minutes.

In [ ]:
# Install everything we need
!pip install datasets transformers torch scikit-learn pandas matplotlib seaborn -q
print('All libraries installed!')

## Step 2 — Load the dataset
We use the PhiUSIIL dataset from HuggingFace. It has 235,000 labeled URLs — phishing or legitimate.

In [ ]:
from datasets import load_dataset
import pandas as pd

print('Loading PhiUSIIL phishing dataset from HuggingFace...')
print('This may take 1-2 minutes on first run.')

dataset = load_dataset('ealvaradob/phishing-dataset', 'urls')

print(f'Dataset loaded!')
print(f'Training samples: {len(dataset["train"])}')
print(f'Columns: {dataset["train"].column_names}')

## Step 3 — Explore the data
Always look at your data before training. Know what you are working with.

In [ ]:
# Convert to pandas for easy exploration
df = pd.DataFrame(dataset['train'])

print('=== First 5 rows ===')
print(df.head())

print('\n=== Label distribution ===')
print(df['label'].value_counts())
print(f'\n0 = legitimate URL')
print(f'1 = phishing URL')

print('\n=== Sample legitimate URLs ===')
print(df[df['label']==0]['text'].sample(5).tolist())

print('\n=== Sample phishing URLs ===')
print(df[df['label']==1]['text'].sample(5).tolist())

## Step 4 — Visualize the data
A simple chart to show the balance between phishing and legitimate URLs.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Label distribution
label_counts = df['label'].value_counts()
axes[0].bar(['Legitimate', 'Phishing'], label_counts.values, color=['#4CAF50', '#F44336'])
axes[0].set_title('Label Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 500, str(v), ha='center', fontweight='bold')

# URL length distribution
df['url_length'] = df['text'].str.len()
axes[1].hist(df[df['label']==0]['url_length'], bins=50, alpha=0.7, color='#4CAF50', label='Legitimate')
axes[1].hist(df[df['label']==1]['url_length'], bins=50, alpha=0.7, color='#F44336', label='Phishing')
axes[1].set_title('URL Length Distribution')
axes[1].set_xlabel('URL Length (characters)')
axes[1].legend()

plt.tight_layout()
plt.savefig('dataset_exploration.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as dataset_exploration.png')

## Step 5 — Clean the data
Remove bad rows, duplicates, and URLs that are too short to be useful.

In [ ]:
print(f'Before cleaning: {len(df)} rows')

# Remove duplicates
df = df.drop_duplicates(subset='text')
print(f'After removing duplicates: {len(df)} rows')

# Remove empty or very short URLs
df = df[df['text'].str.len() > 5]
print(f'After removing short URLs: {len(df)} rows')

# Remove nulls
df = df.dropna(subset=['text', 'label'])
print(f'After removing nulls: {len(df)} rows')

# Make sure labels are integers
df['label'] = df['label'].astype(int)

print('\nCleaning complete!')
print(f'Final label distribution:')
print(df['label'].value_counts())

## Step 6 — Split into train / validation / test sets
Standard ML practice: 80% train, 10% validation, 10% test.

In [ ]:
from sklearn.model_selection import train_test_split

# We use a sample of 50,000 for faster training - still very good results
# You can increase this if you have more time
SAMPLE_SIZE = 50000

# Sample evenly from both classes
df_legit = df[df['label']==0].sample(SAMPLE_SIZE//2, random_state=42)
df_phish = df[df['label']==1].sample(SAMPLE_SIZE//2, random_state=42)
df_sample = pd.concat([df_legit, df_phish]).sample(frac=1, random_state=42)  # shuffle

# Split
X = df_sample['text'].tolist()
y = df_sample['label'].tolist()

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f'Training set:   {len(X_train)} samples')
print(f'Validation set: {len(X_val)} samples')
print(f'Test set:       {len(X_test)} samples')
print(f'\nTotal: {len(X_train)+len(X_val)+len(X_test)} samples')

## Step 7 — Save everything to Google Drive
Save the processed data so you don't have to redo this tomorrow.

In [ ]:
from google.colab import drive
import json
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create a folder for this project
SAVE_PATH = '/content/drive/MyDrive/privaware-v2/'
os.makedirs(SAVE_PATH, exist_ok=True)

# Save train/val/test as JSON
for split_name, X_split, y_split in [('train', X_train, y_train), 
                                       ('val', X_val, y_val), 
                                       ('test', X_test, y_test)]:
    data = [{'text': x, 'label': y} for x, y in zip(X_split, y_split)]
    with open(f'{SAVE_PATH}phishing_{split_name}.json', 'w') as f:
        json.dump(data, f)
    print(f'Saved {split_name} set ({len(data)} samples)')

print(f'\nAll files saved to Google Drive at: {SAVE_PATH}')
print('You are done for today!')

## Step 8 — Quick sanity check
Let's make sure the saved files look correct.

In [ ]:
# Load one file back and check it
with open(f'{SAVE_PATH}phishing_train.json', 'r') as f:
    loaded = json.load(f)

print('Sample from saved training data:')
for item in loaded[:3]:
    label_name = 'PHISHING' if item['label'] == 1 else 'LEGITIMATE'
    print(f'  [{label_name}] {item["text"]}')

print(f'\nTotal training samples in file: {len(loaded)}')
print('\nDAY 1 COMPLETE!')
print('Tomorrow: open Day2_Train_Phishing.ipynb and train the model.')

---
## Summary of what you did today
- Loaded 235,000 labeled URLs from PhiUSIIL dataset
- Explored and visualized the data
- Cleaned: removed duplicates, nulls, short URLs
- Sampled 50,000 balanced examples (25k phishing, 25k legit)
- Split into 80/10/10 train/val/test
- Saved to Google Drive

## What to note down for the panel presentation
- Dataset: PhiUSIIL (235,000 labeled URLs)
- Your sample size: 50,000
- Class balance: 50% phishing, 50% legitimate (balanced dataset)
- Split: 80% train (40,000) / 10% val (5,000) / 10% test (5,000)
